# 1. Establish a set of transforms in a standard, interpretable input structure

In [ ]:
wg_sets = [
    {
        "description": "First spatial lag, group concentric rings",
        "variables": [
            "y",
            "k",
            "l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "All combinations of second-order group spatial lags",
        "variables": [
            "wg1_k", "wg1_l",
            "wg2_k", "wg2_l",
            "wg3_k", "wg3_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8",
            "g2": {
                "value": "pc4",
                "inner": ["pc8"]
            },
            "g3": {
                "value": "ttwa",
                "inner": ["pc8", "pc4"]
            }
        }
    },
    {
        "description": "Third and 4th order group lags for wg1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "g1": "pc8"
        }
    },
    {
        "description": "Local and global FE transforms for w2g1",
        "variables": [
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-wg1)": {
                "i_minus": True,
                "value": "pc8"
            },
            "(i-vg1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "pc8"
            }
        }
    },
    {
        "description": "Local and global transforms for g1",
        "variables": [
            "y", "k", "l",
            "wg1_y", "wg1_k", "wg1_l",
            "w2g1_k", "w2g1_l",
            "w3g1_k", "w3g1_l",
            "w4g1_k", "w4g1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-wg1)": {
                "i_minus": True,
                "value": "pc8"
            },
            "(i-vg1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "pc8"
            }
        }
    }
]

wd_sets = [
    {
        "description": "Distance-weighted spatial lags",
        "variables": [
            "y",
            "k",
            "l"
        ],
        "type": "n",
        "transforms": [
            "d1",
            "d2",
            "d3"
        ]
    },
    {
        "description": "Higher-index d1 lags",
        "variables": [
            "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l"
        ],
        "type": "n",
        "transforms": [
            "d1"
        ]
    },
    {
        "description": "Higher-index d2 lags",
        "variables": [
            "wd2_k", "wd2_l"
        ],
        "type": "n",
        "transforms": [
            "d2"
        ]
    },
    {
        "description": "Higher-index d3 lags",
        "variables": [
            "wd3_k", "wd3_l"
        ],
        "type": "n",
        "transforms": [
            "d3"
        ]
    },
    {
        "description": "Local transforms for w2d1",
        "variables": [
            "y", "k", "l",
            "wd1_y", "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "wd2_k", "wd2_l",
            "w2d1_k", "w2d1_l",
            "wd3_k", "wd3_l",
            "w3d1_k", "w3d1_l",
            "w4d1_k", "w4d1_l"
        ],
        "type": "n",
        "transforms": {
            "(i-wd1)": {
                "i_minus": True,
                "value": "d1"
            }
        }
    },
    {
        "description": "Global transforms for d1",
        "variables": [
            "y", "k", "l",
            "wd1_y", "wd1_k", "wd1_l",
            "w2d1_k", "w2d1_l",
            "w3d1_k", "w3d1_l",
            "w4d1_k", "w4d1_l"
        ],
        "type": "g",
        "transforms": {
            "(i-vd1)": {
                "i_minus": True,
                "leave_one_out": False,
                "value": "ttwa"
            }
        }
    }
]

# 2. Parse many-to-many relation to a one-to-one relation

In [2]:
import re

# Parses and generates the output column name based on transformation rules.
# Automatically increments indices (e.g., w2g1 -> w3g1) if the transformations match.
def get_out_name(var_name: str, transform_name: str) -> str:
    if "_" in var_name:
        parts = var_name.split("_", 1)
        prefix = parts[0]
        suffix = parts[1]
        
        # Check if the last transformation matches the new one exactly
        match = re.search(r'w(\d+)?' + re.escape(transform_name) + r'$', prefix)

        if transform_name.startswith("(") and transform_name.endswith(")"):
            new_prefix = f"{transform_name}{prefix}"
        elif match:
            num = int(match.group(1)) if match.group(1) else 1
            new_prefix = prefix[:match.start()] + f"w{num + 1}{transform_name}"
        else:
            new_prefix = f"w{transform_name}{prefix}"
            
        return f"{new_prefix}_{suffix}"
    else:
        # Base variable renaming (e.g., 'y' -> 'wg1_y')
        if transform_name.startswith("(") and transform_name.endswith(")"):
            return f"{transform_name}_{var_name}"
        else:
            return f"w{transform_name}_{var_name}"

transform_tree = { "g": [], "n": [] }
mapping_dict = { "g": {}, "n": {} }
for seq_type, w_sets in { "g": wg_sets, "n": wd_sets }.items():
    for w_set in w_sets:
        variables = w_set["variables"]
        transforms = w_set["transforms"]
        w_type = w_set["type"]
        
        # Normalize inputs to dictionaries for uniform loop processing
        if isinstance(transforms, list):
            transforms = {t: t for t in transforms}
        if isinstance(variables, list):
            variables = {v: v for v in variables}

        for new_base, current_col in variables.items():
            for t_name, t_val in transforms.items():
                
                # Parse output string
                out_col = get_out_name(new_base, t_name)
                
                # Parse transformation attributes
                w_col: str                      = t_val
                i_minus: bool                   = False
                leave_one_out: bool             = True
                is_hybrid: bool                 = False
                inner_cols: list[str] | None    = None
                if isinstance(t_val, dict):                
                    weight_val = t_val.get("value")
                    if type(weight_val) is not str:
                        raise ValueError(f"Expected string for weight value, got {type(weight_val)}: {weight_val}")
                    w_col = weight_val
                    i_minus = t_val.get("i_minus", False)
                    inner_cols = t_val.get("inner", None)
                    leave_one_out = t_val.get("leave_one_out", True)
                    is_hybrid = "leave_one_out" in t_val  # Triggers overlapping individual graph cluster logic
                
                # Dispatch to appropriate mathematical pipeline 
                if w_type == "g":
                    tree_obj = {
                        "input_col": current_col,
                        "group_col": w_col,
                        "out_col": out_col,
                        "inner_cols": inner_cols,
                        "leave_one_out": leave_one_out,
                        "i_minus": i_minus,
                        "type": w_type
                    }
                elif w_type == "n":
                    tree_obj = {
                        "input_col": current_col,
                        "weight_col": w_col,
                        "out_col": out_col,
                        "leave_one_out": leave_one_out,
                        "i_minus": i_minus,
                        "type": w_type
                    }
                else:
                    raise ValueError(f"Unknown weight type: {w_type}")
                
                if seq_type == "g":
                    transform_tree["g"].append(tree_obj)
                    mapping_dict["g"][out_col] = current_col
                elif seq_type == "n":
                    transform_tree["n"].append(tree_obj)
                    mapping_dict["n"][out_col] = current_col

# Add depth property to each entry in transformation tree
# Based on following the mapping_dict backwards from each out_col to its input_col, counting the number of steps until reaching a base variable (not in mapping_dict).
for w_type in transform_tree:
    for transform in transform_tree[w_type]:
        depth = 1
        current_col = transform["input_col"]
        while current_col in mapping_dict[w_type]:
            current_col = mapping_dict[w_type][current_col]
            depth += 1
        transform["depth"] = depth

print("Transformation tree built successfully.")
# Output transformation tree as json for inspection
import json
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
outfile = dirs.tmp_dir / "transform_tree.json"
with open(outfile, "w") as f:
    json.dump(transform_tree, f, indent=4)

# For each depth level, list the variable-transformation pairs that will take place at that depth.
for w_type in transform_tree:
    max_depth = max(t["depth"] for t in transform_tree[w_type]) if transform_tree[w_type] else 0
    print(f"\n{w_type}-type transformations by depth:")
    for depth in range(1, max_depth + 1):
        depth_transforms = [t for t in transform_tree[w_type] if t["depth"] == depth]
        print(f"Depth {depth}: {[f'{t['input_col']} -> {t['out_col']}' for t in depth_transforms]}")

Transformation tree built successfully.

g-type transformations by depth:
Depth 1: ['y -> wg1_y', 'y -> wg2_y', 'y -> wg3_y', 'k -> wg1_k', 'k -> wg2_k', 'k -> wg3_k', 'l -> wg1_l', 'l -> wg2_l', 'l -> wg3_l', 'y -> (i-wg1)_y', 'y -> (i-vg1)_y', 'k -> (i-wg1)_k', 'k -> (i-vg1)_k', 'l -> (i-wg1)_l', 'l -> (i-vg1)_l']
Depth 2: ['wg1_k -> w2g1_k', 'wg1_k -> wg2wg1_k', 'wg1_k -> wg3wg1_k', 'wg1_l -> w2g1_l', 'wg1_l -> wg2wg1_l', 'wg1_l -> wg3wg1_l', 'wg2_k -> wg1wg2_k', 'wg2_k -> w2g2_k', 'wg2_k -> wg3wg2_k', 'wg2_l -> wg1wg2_l', 'wg2_l -> w2g2_l', 'wg2_l -> wg3wg2_l', 'wg3_k -> wg1wg3_k', 'wg3_k -> wg2wg3_k', 'wg3_k -> w2g3_k', 'wg3_l -> wg1wg3_l', 'wg3_l -> wg2wg3_l', 'wg3_l -> w2g3_l', 'wg1_y -> (i-wg1)wg1_y', 'wg1_y -> (i-vg1)wg1_y', 'wg1_k -> (i-wg1)wg1_k', 'wg1_k -> (i-vg1)wg1_k', 'wg1_l -> (i-wg1)wg1_l', 'wg1_l -> (i-vg1)wg1_l']
Depth 3: ['w2g1_k -> w3g1_k', 'w2g1_l -> w3g1_l', 'w2g1_k -> (i-wg1)w2g1_k', 'w2g1_k -> (i-vg1)w2g1_k', 'w2g1_l -> (i-wg1)w2g1_l', 'w2g1_l -> (i-vg1)w2g1_l'

In [ ]:
import ibis
from ibis import _
import pandas as pd
import itertools
from utils.f_0_dirs import get_data_dirs

# Batches multiple group-based spatial lags and processes them in a single Ibis mutation.
# Calculates mutually exclusive donut holes (Inclusion-Exclusion) and (I-W) fixed effects.
def apply_group_W(t_panel: ibis.Table, transforms: list[dict]) -> ibis.Table:

    new_cols = {}
    
    for trans_def in transforms:
        input_col = trans_def["input_col"]
        group_col = trans_def["group_col"]
        out_col = trans_def["out_col"]
        inner_cols = trans_def.get("inner_cols")
        leave_one_out = trans_def.get("leave_one_out", True)
        i_minus = trans_def.get("i_minus", False)

        # 1. Base group sum and count
        sum_col = _[input_col].sum().over(group_by=[_[group_col], _.year])
        count_col = _[input_col].count().over(group_by=[_[group_col], _.year])
        
        # 2. Subtract inner groups if specified (Inclusion-Exclusion Principle)
        if not inner_cols:
            # 3. Standard group (no donut hole)
            if leave_one_out:
                numerator = sum_col - ibis.coalesce(_[input_col], 0)
                denominator = count_col - 1
            else:
                numerator = sum_col
                denominator = count_col
        else:
            sub_sum = None
            sub_count = None
            
            for r in range(1, len(inner_cols) + 1):
                sign = 1 if r % 2 != 0 else -1
                for combo in itertools.combinations(inner_cols, r):
                    part_cols = [_[group_col], _.year] + [_[c] for c in combo]
                    
                    term_sum = _[input_col].sum().over(group_by=part_cols)
                    term_count = _[input_col].count().over(group_by=part_cols)
                    
                    if sub_sum is None:
                        sub_sum = ibis.coalesce(term_sum, 0)
                        sub_count = ibis.coalesce(term_count, 0)
                    else:
                        sub_sum = sub_sum + (ibis.coalesce(term_sum, 0) * sign)         # type: ignore
                        sub_count = sub_count + (ibis.coalesce(term_count, 0) * sign)   # type: ignore
            
            base_sum = sum_col - sub_sum
            base_count = count_col - sub_count
            
            if leave_one_out:
                numerator = base_sum
                denominator = base_count
            else:
                numerator = base_sum + ibis.coalesce(_[input_col], 0)
                denominator = base_count + 1                
            
        # 4. Row-normalize and apply (I - W) logic
        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        new_cols[out_col] = _[input_col] - spatial_lag if i_minus else spatial_lag

    # Apply all transformations at this depth in one deferred AST branch
    return t_panel.mutate(**new_cols)

# Batches multiple distance-decay and overlapping individual network group lags.
# Executes a single structural join per depth level.
def apply_network_W(t_panel: ibis.Table, t_distance: ibis.Table, transforms: list[dict]) -> ibis.Table:

    input_cols = list(set(t["input_col"] for t in transforms))

    # 1. Build deferred selections to isolate target and peer data dynamically
    target_selects = {"target_firm": _.registered_number, "target_year": _.year}
    peer_selects = {"peer_firm": _.registered_number, "peer_year": _.year}

    for col in input_cols:
        target_selects[f"target_{col}"] = _[col]
        peer_selects[f"peer_{col}"] = _[col]

    t_target = t_panel.select(**target_selects)
    t_peer = t_panel.select(**peer_selects)

    # 2. Single Master Join (Chain safely evaluates `_` as the growing left-side table)
    t_joined = (
        t_distance
        .inner_join(t_target, _.firm_i == t_target.target_firm)                                     # type: ignore
        .inner_join(t_peer, (_.firm_j == t_peer.peer_firm) & (_.target_year == t_peer.peer_year))   # type: ignore
    )

    # 3. Build deferred aggregation dictionary for all transforms concurrently
    agg_exprs = {}
    for transf_def in transforms:
        in_col = transf_def["input_col"]
        w_col = transf_def["weight_col"]
        out_col = transf_def["out_col"]
        i_minus = transf_def.get("i_minus", False)

        target_val = _[f"target_{in_col}"]
        peer_val = _[f"peer_{in_col}"]
        weight_val = _[w_col]

        # Continuous network distance decay
        weighted_val = peer_val * weight_val
        numerator = weighted_val.sum()
        denominator = ibis.ifelse(peer_val.notnull(), weight_val, ibis.null()).sum()        # type: ignore

        spatial_lag = ibis.ifelse(denominator > 0, numerator / denominator, ibis.null())
        agg_exprs[out_col] = target_val.first() - spatial_lag if i_minus else spatial_lag

    # 4. Perform the massive single aggregation
    t_distance_weighted = (
        t_joined
        .group_by([_.firm_i, _.target_year])
        .aggregate(**agg_exprs)
    )

    # 5. Join back and drop redundant identifiers cleanly
    t_panel_mutated = (
        t_panel
        .left_join(
            t_distance_weighted,
            (_.registered_number == t_distance_weighted.firm_i) & (_.year == t_distance_weighted.target_year)   # type: ignore
        )
        .drop("firm_i", "target_year")
    )
    
    return t_panel_mutated


# ==========================================
# Schema Iteration & Table Management
# ==========================================

# Assuming 'transform_tree' contains your parsed JSON dict
# transform_tree = json.loads(json_string)

panel_name = "working_yearly"
fixed_name = "working_fixed"
distance_name = "working_distance_ttwa_km"

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

table_panel = (
    con.table(panel_name)
    .select("registered_number", "year", "gva1", "total_assets", "employees")
    .rename({"gva": "gva1"})
    .distinct(on=["registered_number", "year"])
    .filter(
        (_["gva"].notnull())            & (_["gva"] > 0) &
        (_["total_assets"].notnull())   & (_["total_assets"] > 0) &
        (_["employees"].notnull())      & (_["employees"] > 0)
    )
    .mutate(
        y = _['gva'].log(),
        k = _['total_assets'].log(),
        l = _['employees'].log()
    )
)

table_fixed = (
    con.table(fixed_name)
    .select("registered_number", "pc8", "pc4", "ttwa")
    .distinct(on="registered_number")
)

# Constrain the universe of distances to only valid firms mapped in the dataset
table_uniques = (
    table_panel
    .inner_join(table_fixed, "registered_number")
    .distinct(on="registered_number")
    .select("registered_number")
)

table_distance_raw = (
    con.table(distance_name)
    .inner_join(table_uniques, _.firm_i == table_uniques.registered_number)
    .filter(_.distance_meters.notnull())
)

should_run = 'ns' # Options: 'g', 'n', 'no', 'ns' 'all'

# 1m14s run
if should_run == 'g' or should_run == 'all':
    # Run groups
    t_current_g = table_panel.left_join(table_fixed, "registered_number")
    max_depth_g = max(t.get("depth", 1) for t in transform_tree.get("g", []))
    print(f"\nBeginning group transformations, {len(transform_tree.get('g', []))} required across depth {max_depth_g}.")
    for d in range(1, max_depth_g + 1):
        g_transforms = [t for t in transform_tree.get("g", []) if t.get("depth") == d]
        print(f"--- applying depth {d} with {len(g_transforms)} transformations, {len(t_current_g.columns)} columns...")
        if not g_transforms:
            print(f"No group transformations found at depth {d}.")
            continue
        t_current_g = apply_group_W(t_current_g, g_transforms)
    t_final_g = (
        t_current_g
        .drop("registered_number_right", "pc8", "pc4", "ttwa")
    )
    con.create_table("working_yearly_g", t_final_g, overwrite=True)
    print(con.table("working_yearly_g").sample(0.001).execute())

# 1m24s for depth: 1
# 10 minute run, out of memory
# 2m30s per depth. Should be around 7 minutes total
n_runs = []
if should_run == 'no' or should_run == 'ns' or should_run == 'all':
    n_runs.append('no')
if should_run == 'n' or should_run == 'ns' or should_run == 'all':
    n_runs.append('n')

for run_type in n_runs:
    # Run networks
    time_start = pd.Timestamp.now()

    # Calculate the distance table
    table_distance = (
        table_distance_raw
        .filter(_.distance_meters >= 1 if run_type == 'no' else True)
        .mutate(
            d1 = 1 / (_.distance_meters + 1),
            d2 = 1 / (_.distance_meters + 1) ** 2,
            d3 = (-_.distance_meters / 1000).exp()
        )
    )

    t_start_n = (
        table_panel
        .left_join(table_fixed, "registered_number")
        .drop("registered_number_right")
    )
    max_depth_n = max(t.get("depth", 1) for t in transform_tree.get("n", []))
    print(f"\nBeginning network transformations writing to db at each step, {len(transform_tree.get('n', []))} required across depth {max_depth_n}.")
    write_name = f"working_yearly_{run_type}"
    
    for d in range(1, max_depth_n + 1):
        try:
            t_imported_n = con.table(write_name)
            t_imported_rows = t_imported_n.count().execute()
            t_imported_columns = t_imported_n.columns
            if t_imported_rows == 0 or len(t_imported_columns) < 5:
                raise ValueError(f"Imported table {write_name} is empty or has insufficient columns.")
            t_current_n = (
                t_imported_n
                .left_join(table_fixed, "registered_number")
                .drop("registered_number_right")
            )
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- got table from {write_name} with shape ({t_imported_rows}, {len(t_imported_columns)})")
        except Exception as e:
            t_current_n = t_start_n
            time_check = (pd.Timestamp.now() - time_start).total_seconds()
            print(f"[{time_check:.1f}s] --- starting from t_start_n with shape ({t_start_n.count().execute()}, {len(t_start_n.columns)}) for depth {d} due to: {e}")

        # Run network transforms
        n_transforms = [t for t in transform_tree.get("n", []) if (t.get("depth") == d) and (t.get("type") == "n")]
        time_check = (pd.Timestamp.now() - time_start).total_seconds()
        print(f"[{time_check:.1f}s] --- applying depth {d} with {len(n_transforms)} network transformations, {len(t_current_n.columns)} columns...")
        if not n_transforms:
            print(f"No network transformations found at depth {d}.")
        else:
            t_current_n = apply_network_W(t_current_n, table_distance, n_transforms)
   
        # Run groups
        g_transforms = [t for t in transform_tree.get("n", []) if (t.get("depth") == d) and (t.get("type") == "g")]
        time_check = (pd.Timestamp.now() - time_start).total_seconds()
        print(f"[{time_check:.1f}s] --- applying depth {d} with {len(g_transforms)} group transformations, {len(t_current_n.columns)} columns...")
        if not g_transforms:
            pass
        else:
            t_current_n = apply_group_W(t_current_n, g_transforms)

        # Output the table to the database, dropping redundant columns
        t_out_n = t_current_n
        for col in ["registered_number_right", "pc8", "pc4", "ttwa"]:
            if col in t_out_n.columns:
                t_out_n = t_out_n.drop(col)
        con.create_table(write_name, t_out_n, overwrite=True)

    print(con.table(write_name).sample(0.001).execute())



Beginning network transformations writing to db at each step, 2 required across depth 1.
[0.4s] --- got table from working_yearly_no with shape (1085633, 49)
[0.4s] --- applying depth 1 with 2 network transformations, 52 columns...
[0.4s] --- applying depth 1 with 0 group transformations, 54 columns...


RuntimeError: Query interrupted

In [18]:
# %%script drop table

# Drop table "working_yearly_n" if it exists
import ibis
from utils.f_0_dirs import get_data_dirs
dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

table_name = "working_yearly_g"
drop = False
if drop:
    if con.table(table_name).columns:
        con.drop_table(table_name)
        print(f"Dropped table '{table_name}' after processing.")

merge = False
if merge:
    # Add the ttwa column from table_fixed to table_name table using a "registered_number" join
    # t_fixed_ttwa = (
    #     con.table("working_fixed")
    #     .select("registered_number", "ttwa")
    # )
    t_panel_ttwa = (
        con.table(table_name)
        # .left_join(t_fixed_ttwa, "registered_number")
        .drop("ttwa")
    )
    con.create_table(table_name, t_panel_ttwa, overwrite=True)

drop_columns = False
if drop_columns:
    # Drop columns that start with (i-vd1) or (i-wd1)wd2 or (i-wd1)wd3
    t_panel = con.table(table_name)
    drop_patterns = ["(i-vd1)", "(i-wd1)"]
    drop_these_columns = [col for col in t_panel.columns if any(col.startswith(pattern) for pattern in drop_patterns)]
    t_panel_dropped = (
        con.table(table_name)
        .drop(*drop_these_columns)
    )
    # print(t_panel_dropped.columns)
    con.create_table(table_name, t_panel_dropped, overwrite=True)

rename = True
if rename:
    for table_name in ["working_yearly_g", "working_yearly_n", "working_yearly_no"]:
        in_table = con.table(table_name)

        rename_dict = {}
        for_leading_w = True
        if for_leading_w:
            valid_cols = [col for col in in_table.columns if col.startswith("w(")]
            for col in valid_cols:
                new_col_name = col[1:]  # Remove the leading 'w'
                rename_dict[new_col_name] = col

        for_no_underscore = False
        if for_no_underscore:
            for col in in_table.columns:
                if len(col) == 1:
                    continue
                if not col.endswith("k") and not col.endswith("l") and not col.endswith("y"):
                    continue
                if col.endswith("_k") or col.endswith("_l") or col.endswith("_y"):
                    continue
                # Remove the last character from the col name for prefix
                col_prefix = col[:-1]
                col_suffix = col[-1]
                new_col_name = f"{col_prefix}_{col_suffix}"
                rename_dict[new_col_name] = col

        print(rename_dict)


        out_table = (
            in_table
            .rename(rename_dict)
        )       
        print(out_table.columns)
        con.create_table(table_name, out_table, overwrite=True)

{'(i-wg1)_y': 'w(i-wg1)_y', '(i-vg1)_y': 'w(i-vg1)_y', '(i-wg1)_k': 'w(i-wg1)_k', '(i-vg1)_k': 'w(i-vg1)_k', '(i-wg1)_l': 'w(i-wg1)_l', '(i-vg1)_l': 'w(i-vg1)_l'}
('registered_number', 'year', 'gva', 'total_assets', 'employees', 'y', 'k', 'l', 'wg1_y', 'wg2_y', 'wg3_y', 'wg1_k', 'wg2_k', 'wg3_k', 'wg1_l', 'wg2_l', 'wg3_l', '(i-wg1)_y', '(i-vg1)_y', '(i-wg1)_k', '(i-vg1)_k', '(i-wg1)_l', '(i-vg1)_l', 'w2g1_k', 'wg2wg1_k', 'wg3wg1_k', 'w2g1_l', 'wg2wg1_l', 'wg3wg1_l', 'wg1wg2_k', 'w2g2_k', 'wg3wg2_k', 'wg1wg2_l', 'w2g2_l', 'wg3wg2_l', 'wg1wg3_k', 'wg2wg3_k', 'w2g3_k', 'wg1wg3_l', 'wg2wg3_l', 'w2g3_l', '(i-wg1)wg1_y', '(i-vg1)wg1_y', '(i-wg1)wg1_k', '(i-vg1)wg1_k', '(i-wg1)wg1_l', '(i-vg1)wg1_l', 'w3g1_k', 'w3g1_l', '(i-wg1)w2g1_k', '(i-vg1)w2g1_k', '(i-wg1)w2g1_l', '(i-vg1)w2g1_l', 'w4g1_k', 'w4g1_l', '(i-wg1)w3g1_k', '(i-vg1)w3g1_k', '(i-wg1)w3g1_l', '(i-vg1)w3g1_l', '(i-wg1)w4g1_k', '(i-vg1)w4g1_k', '(i-wg1)w4g1_l', '(i-vg1)w4g1_l')
{'(i-vd1)_y': 'w(i-vd1)_y', '(i-vd1)_k': 'w(i-vd1)_k'

# 2. Read and sample to verify

In [ ]:
t_distance = con.table("working_yearly_g")
out_file = dirs.tmp_dir / "w_transforms_sample_d.xlsx"
print(t_distance.limit(10).execute())
t_distance.sample(0.001).execute().to_excel(out_file, index=False)
print(f"Exported check to {out_file}")

  registered_number  year          gva  total_assets  employees         y  \
0          04464220  2009  5062.719843   5996.177443        134  8.529659   
1          11425513  2020  2563.924112   9709.478507         70  7.849294   
2          02132170  2011   927.324632    249.451760         13  6.832304   
3          00598760  2008  2923.641856   2940.254888         54  7.980585   
4          06228171  2011    97.563919    113.939902         10  4.580508   
5          02994954  2017   353.990178    921.934183         23  5.869269   
6          01481734  2017  4477.377772   9023.323605         31  8.406793   
7          01481734  2022  2653.136770   6582.900621         32  7.883498   
8          05938886  2022  6673.480946  12791.429647        113  8.805897   
9          02994954  2022   340.432313   5846.362364         23  5.830216   

          k         l     wg1_y     wg2_y  ...  (i-wg1)w2g1_k  (i-vg1)w2g1_k  \
0  8.698877  4.897840       NaN  8.806353  ...            NaN           

In [ ]:
# Load table working_yearly_g.
# Create summary statistics:
# Take the mean, median, standard deviation, max, min, max (% of median), min (% of median) for each column in the table.
import ibis
from ibis import _, selectors as s
import pandas as pd
import xlsxwriter

from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="calculations")
con = ibis.duckdb.connect(dirs.db_path)

out_file = dirs.output_dir / "w_transforms_summ.xlsx"
g_name = "working_yearly_g"
n_name = "working_yearly_n"
no_name = "working_yearly_no"

with pd.ExcelWriter(out_file, engine="xlsxwriter") as writer:
    workbook  = writer.book
    wrap_format = workbook.add_format({'text_wrap': True, 'align': 'center'})
    header_col_format = workbook.add_format({'valign': 'vcenter'})

    for table_name in [g_name, n_name, no_name]:

        this_table = con.table(table_name)

        use_columns = [col for col in this_table.columns if col not in ["registered_number", "year", "pc8", "pc4", "ttwa"]]

        t_summary = (
            this_table
            .select(use_columns)
            .rename({
                "y": "gva",
                "k": "total_assets",
                "l": "employees"
            })
            .aggregate(
                s.across(
                    s.all(),
                    {
                        "mean": _.mean(),
                        "median": _.median(),
                        "std": _.std(),
                        "max": _.max(),
                        "min": _.min(),
                    }
                )
            )
            .pivot_longer(
                s.all(),
                names_to=["variable", "statistic"],
                names_pattern=r"^(.*)_(.*)$",
                values_to="value"
            )
            .pivot_wider(
                names_from="statistic",
                values_from="value"
            )
            .mutate(
                max_percent_mean = ibis.ifelse(_.mean.abs() >= 1, _.max / _.mean - 1, None),
                min_percent_mean = ibis.ifelse(_.mean.abs() >= 1, _.min / _.mean - 1, None),
                # Split the variable name by _ and take the last part as suffix (there may be only one item)
                prefix = _.variable.split("_")[0],
                suffix=_.variable.split("_")[-1]
            )
        )

        # Display in dataframe
        # take every t_column, separate by _ the suffix will be the new column of our dataframe
        # the prefix will be our row of our dataframe.
        # For every cell in our dataframe, insert the t_column's mean and standard deviation in that column
        # As a string, joined by (\n), the standard deviation should be in brackets underneath
        t_display = (
            t_summary
            .select("mean", "std", "prefix", "suffix")
            .mutate(
                prefix_mod=_.prefix.re_replace(r"^(?:y|k|l)$", "base"),
            )
            .drop("prefix")
            .pivot_wider(
                names_from="suffix",
                values_from=["mean", "std"]
            )
        )
        df_display = t_display.execute()
        # Format mean and std pairs into a combined string per column
        for val in ["y", "k", "l"]:
            mean_col = f"mean_{val}"
            std_col = f"std_{val}"
            if mean_col in df_display.columns and std_col in df_display.columns:
                df_display[val] = [
                    f"{m:,.3f}\n({s:,.3f})" if pd.notnull(m) and pd.notnull(s) else None
                    for m, s in zip(df_display[mean_col], df_display[std_col])
                ]
                df_display = df_display.drop(columns=[mean_col, std_col])
        # Sort the dataframe by prefix_mod, according to the index
        # of which transform it is in the transform_tree.
        sort_key = table_name.split("_")[-1]
        if sort_key == "no":
            sort_key = "n"
        sort_list = [t["out_col"].split("_")[0] for t in transform_tree[sort_key]]
        sort_list_unique = list(dict.fromkeys(sort_list))
        sort_list_unique.insert(0, "base")
        sort_order = {t: i for i, t in enumerate(sort_list_unique)}
        df_sorted = (
            df_display
            .sort_values(by="prefix_mod", key=lambda x: x.map(sort_order), ignore_index=True)
        )
        display(df_sorted)

        # Write as a tab to excel file
        desc_dirs = get_data_dirs(segment="descriptives")
        df_sorted.to_excel(writer, sheet_name=table_name, index=False)
        work_sheet = writer.sheets[table_name]
        work_sheet.set_column(0, 0, 20, header_col_format)
        work_sheet.set_column(1, 3, 10, wrap_format)
print(f"Exported summary to {out_file}")

,prefix_mod,y,k,l
0,base,8.475\n(1.580),9.443\n(1.900),4.501\n(1.354)
1,wg1,8.711\n(1.120),9.720\n(1.373),4.634\n(0.960)
2,wg2,8.457\n(0.577),9.422\n(0.714),4.483\n(0.399)
3,wg3,8.462\n(0.293),9.430\n(0.365),4.492\n(0.169)
4,w2g1,NaN,9.720\n(1.346),4.634\n(0.936)
5,wg2wg1,NaN,9.684\n(0.738),4.623\n(0.488)
6,wg3wg1,NaN,9.696\n(0.349),4.634\n(0.199)
7,wg1wg2,NaN,9.513\n(0.664),4.508\n(0.369)
8,w2g2,NaN,9.427\n(0.690),4.488\n(0.380)
9,wg3wg2,NaN,9.409\n(0.373),4.475\n(0.162)


,prefix_mod,y,k,l
0,base,8.505\n(1.603),9.478\n(1.924),4.525\n(1.374)
1,wd1,8.610\n(0.913),9.602\n(1.109),4.592\n(0.755)
2,wd2,8.594\n(1.054),9.582\n(1.283),4.578\n(0.876)
3,wd3,8.551\n(0.602),9.531\n(0.740),4.550\n(0.442)
4,w2d1,NaN,9.652\n(1.025),4.619\n(0.700)
5,w3d1,NaN,9.671\n(0.986),4.629\n(0.673)
6,w4d1,NaN,9.685\n(0.961),4.638\n(0.657)
7,w2d2,NaN,9.621\n(1.239),4.600\n(0.848)
8,w2d3,NaN,9.546\n(0.675),4.556\n(0.392)
9,(i-wd1),-0.101\n(1.539),-0.121\n(1.823),-0.064\n(1.367)


,prefix_mod,y,k,l
0,base,8.505\n(1.603),9.478\n(1.924),4.525\n(1.374)
1,wd1,8.517\n(0.566),9.488\n(0.699),4.523\n(0.402)
2,wd2,8.505\n(0.763),9.472\n(0.940),4.507\n(0.585)
3,wd3,8.512\n(0.587),9.483\n(0.723),4.519\n(0.417)
4,w2d1,NaN,9.505\n(0.623),4.533\n(0.340)
5,w3d1,NaN,9.505\n(0.605),4.531\n(0.324)
6,w4d1,NaN,9.510\n(0.591),4.534\n(0.311)
7,w2d2,NaN,9.506\n(0.823),4.529\n(0.495)
8,w2d3,NaN,9.502\n(0.651),4.530\n(0.359)
9,(i-wd1),-0.008\n(1.598),-0.008\n(1.907),0.004\n(1.396)


Exported summary to C:\Users\lazyst\Files\ucl\Dissertation\calculations\output\w_transforms_summ.xlsx


c:\Users\lazyst\Files\ucl\Dissertation\.venv-main\Lib\site-packages\xlsxwriter\workbook.py:404: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [2]:
import ibis
from utils.f_0_dirs import get_data_dirs
db_path = get_data_dirs().output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(db_path)

con.raw_sql("CHECKPOINT;")
con.disconnect()